<div style="text-align: center;">
  <span style="color: lightskyblue; font-weight: bold; font-size: 50px;">
    Test File for Depth Camera (D435i)
  </span>
</div>

<span style="color: magenta; font-weight: bold; font-size: 30px;">
Coordinate Information
</span>

<div style="color: white; font-size: 15px; line-height: 1.6;">

  <b>Image / Depth Coordinate Convention</b>

  <table style="border-collapse: collapse; margin-top: 5px; color: white;">
    <tr>
      <td style="padding: 4px 14px 4px 2;"><b>Pixel</b></td>
      <td><code>(u, v)</code></td>
    </tr>
    <tr>
      <td style="padding: 4px 14px 4px 2;"><b>Pixel origin</b></td>
      <td>Top-left corner of the image (= Camera optical center)</td>
    </tr>
    <tr>
      <td style="padding: 4px 14px 4px 2;"><b>u-axis</b></td>
      <td>Left → Right</td>
    </tr>
    <tr>
      <td style="padding: 4px 14px 4px 2;"><b>v-axis</b></td>
      <td>Top → Bottom</td>
    </tr>
    <tr>
      <td style="padding: 4px 14px 4px 2;"><b>Depth</b></td>
      <td><code>z</code></td>
    </tr>
    <tr>
      <td style="padding: 4px 14px 4px 2;"><b>3D position</b></td>
      <td><code>(X, Y, Z)</code></td>
    </tr>
    <tr>
      <td style="padding: 4px 14px 4px 2;"><b>3D axes</b></td>
      <td>
        <code>X</code>: Left → Right,
        <code>Y</code>: Top → Bottom,
        <code>Z</code>: Depth direction
      </td>
    </tr>
  </table>

</div>

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Import Library
</span>

In [1]:
import rby1_sdk as rby
import numpy as np
import rby1_lib_ysw as ysw
import logging
import sys
import time
import pyrealsense2 as rs
import cv2

np.set_printoptions(suppress=True, precision=5, linewidth=300)  

# In .ipynb, show logging 
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(message)s",
    # format="%(asctime)s [%(levelname)s] %(message)s",
    stream=sys.stdout,
    force=True,
)

<span style="color:gray; font-weight:bold; font-size:50px;">
Check recognition of D435i and D405
</span>

In [2]:
ctx = rs.context()
devices = ctx.query_devices()

print("Number of RealSense devices:", len(devices))

for i, dev in enumerate(devices):
    name = dev.get_info(rs.camera_info.name)
    serial = dev.get_info(rs.camera_info.serial_number)
    product_line = dev.get_info(rs.camera_info.product_line)
    firmware = dev.get_info(rs.camera_info.firmware_version)
    print(f"[{i}] Name={name}, Serial={serial}, Product Line={product_line}, Firmware={firmware}")

Number of RealSense devices: 3
[0] Name=Intel RealSense D405, Serial=260322279815, Product Line=D400, Firmware=5.15.1.55
[1] Name=Intel RealSense D435I, Serial=250122070658, Product Line=D400, Firmware=5.15.1.55
[2] Name=Intel RealSense D405, Serial=260422275816, Product Line=D400, Firmware=5.15.1.55


<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Initialization of Robot
</span>

In [3]:
robotConfig = ysw.setRobotConfig(ysw.MODE_DIRECT_UPC, ysw.MODEL_A)          # For Real Robot
robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_AND_POWER)
# robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_ONLY)
if (robot != None):
    model = robot.model()

*************************** [Initialization Starts] ****************************

[INFO] SUCCESS: Connected to the Robot!
[INFO] SUCCESS: Power ON the Robot!
[INFO] SUCCESS: Servo ON the Robot!
[INFO] SUCCESS: Enable the Control Manager!
[INFO] SUCCESS: Enable the Grippers!

*************************** [Intialization is Done] ****************************





<span style="color:gray; font-weight:bold; font-size:50px;">
Power OFF
</span>

In [4]:
ysw.powerOFF(robot, robotConfig)

****************************** [powerOFF Starts] *******************************

[INFO] SUCCESS: Disable Grippers!
[INFO] SUCCESS: Disable the Control Manager!

****************************** [PowerOFF is Done] ******************************





True

<span style="color:gray; font-weight:bold; font-size:50px;">
Posture Change
</span>

In [3]:
Desired_Pos = ysw.DEMO_POS     # ZERO_POS, READY_POS, PACK_POS, DEMO_POS
ysw.positionCtrl_joint_22dof(robot, Desired_Pos, minimum_time=5.0, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Initialization of Vision
</span>

In [4]:
d435i = ysw.RealSenseDepthCamera(rgb_on=True, depth_on=True, pixel=(640,480), fps=30, serial_number=ysw.HEAD_D435I_SERIAL)
d435i.initialize()

d405_R = ysw.RealSenseDepthCamera(rgb_on=True, depth_on=True, pixel=(640,480), fps=30, serial_number=ysw.RIGHT_D405_SERIAL)
d405_R.initialize()

d405_L = ysw.RealSenseDepthCamera(rgb_on=True, depth_on=True, pixel=(640,480), fps=30, serial_number=ysw.LEFT_D405_SERIAL)
d405_L.initialize()

<span style="color:green; font-weight:bold; font-size:50px;">
Show ALL Cameras [One D435i / Two D405]
</span> 

In [5]:
"""
# Simple test of 3 RGB-D cameras
# 2 x 3 view
# ESC: Terminate

┌────────────────┬──────────────┬────────────────┐
│ L wrist color  │ Head color   │ R wrist color  │
├────────────────┼──────────────┼────────────────┤
│ L wrist depth  │ Head depth   │ R wrist depth  │
└────────────────┴──────────────┴────────────────┘
"""

camera_dict = {
    # 순서 중요! 위 표와 동일한 순서로
    "wrist_L": d405_L,
    "head": d435i,
    "wrist_R": d405_R,
}

window_name = "RGB-D Cameras: Left Wrist / Head / Right Wrist"

# Display panel size
panel_w = 640
panel_h = 480

def make_blank_panel(text, width=640, height=480):
    image = np.zeros((height, width, 3), dtype=np.uint8)
    cv2.putText(
        image,
        text,
        (30, height // 2),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )
    return image


def resize_and_label(image, label, width=640, height=480):
    if image is None:
        image = make_blank_panel("No Image", width, height)
    else:
        if len(image.shape) == 2:
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

        image = cv2.resize(image, (width, height))

    cv2.putText(
        image,
        label,
        (20, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (0, 255, 0),
        2
    )

    return image



# Start all cameras
started_cameras = []

try:
    for name, cam in camera_dict.items():
        cam.startPipeline()
        started_cameras.append(cam)

    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

    # Fullscreen mode
    cv2.setWindowProperty(
        window_name,
        cv2.WND_PROP_FULLSCREEN,
        cv2.WINDOW_FULLSCREEN
    )

    while True:
        color_panels = []
        depth_panels = []

        for name, cam in camera_dict.items():
            color_frame, depth_frame = cam.getRawFrames()

            if color_frame is None and depth_frame is None:
                color_panel = make_blank_panel(f"{name}: No Frame", panel_w, panel_h)
                depth_panel = make_blank_panel(f"{name}: No Frame", panel_w, panel_h)

            else:
                color_image, depth_image = cam.frameToImage(color_frame, depth_frame)

                if color_image is not None:
                    color_panel = resize_and_label(
                        color_image,
                        f"{name} color",
                        panel_w,
                        panel_h
                    )
                else:
                    color_panel = make_blank_panel(
                        f"{name}: No Color",
                        panel_w,
                        panel_h
                    )

                if depth_image is not None:
                    depth_colorMap = cam.convertDepthToColorMap(depth_image, alpha=0.03)
                    depth_panel = resize_and_label(
                        depth_colorMap,
                        f"{name} depth",
                        panel_w,
                        panel_h
                    )
                else:
                    depth_panel = make_blank_panel(
                        f"{name}: No Depth",
                        panel_w,
                        panel_h
                    )

            color_panels.append(color_panel)
            depth_panels.append(depth_panel)

        top_row = cv2.hconcat(color_panels)
        bottom_row = cv2.hconcat(depth_panels)

        grid_image = cv2.vconcat([top_row, bottom_row])

        cv2.imshow(window_name, grid_image)

        key = cv2.waitKey(1)

        if key == 27:       # ESC key
            break

finally:
    for name, cam in camera_dict.items():
        cam.stopPipeline()
        cam.destroyAllWindows()
    

[INFO] [Left Wrist D405] Pipeline started.
[INFO] [Head D435i] Pipeline started.
[INFO] [Right Wrist D405] Pipeline started.


QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvid

[INFO] [Left Wrist D405] Pipeline stopped.
[INFO] [Head D435i] Pipeline stopped.
[INFO] [Right Wrist D405] Pipeline stopped.


<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Show individual camera [D435i / D405]
</span>

In [ ]:
# Simple test of RGB-D camera
# Click ESC: Terminate

# --- Select camera location ----------------------------
location = "head"       # head / wrist_R / wrist_L
# -------------------------------------------------------

if location == "head":
    cam = d435i
elif location == "wrist_R":
    cam = d405_R
elif location == "wrist_L":
    cam = d405_L
    
cam.startPipeline()

try:
    while True:
        color_frame, depth_frame = cam.getRawFrames()
        color_image, depth_image = cam.frameToImage(color_frame, depth_frame)
        depth_colorMap = cam.convertDepthToColorMap(depth_image, alpha=0.03)
        
        if (cam.rgb_on):
            cam.imshow("color_image", color_image)
        if (cam.depth_on):
            cam.imshow("depth_image", depth_colorMap)
        
        key = cam.waitKey(1)
        if key == 27:       # ESC key
            break
        
finally:
    cam.stopPipeline()
    cam.destroyAllWindows()
        

[INFO] [Right Wrist D405] Pipeline started.
[INFO] [Right Wrist D405] Pipeline stopped.


<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Distance of center pixel
</span>

In [9]:
# Get distance value of center pixel of image
# Click ESC: Terminate

# --- Select camera location ----------------------------
location = "head"       # head / wrist_R / wrist_L
# -------------------------------------------------------

if location == "head":
    cam = d435i
elif location == "wrist_R":
    cam = d405_R
elif location == "wrist_L":
    cam = d405_L
    
cam.startPipeline()

try:
    while True:
        color_frame, depth_frame = cam.getRawFrames()
        
        if color_frame is None or depth_frame is None:
            continue
        
        color_image, depth_image = cam.frameToImage(color_frame, depth_frame)
        depth_colorMap = cam.convertDepthToColorMap(depth_image, alpha=0.03)

        h, w, _ = color_image.shape
        u = w // 2      # origin is left    
        v = h // 2      # origin is top

        distance = cam.getDepth(depth_frame, (u, v))   # unit = [m]

        cv2.circle(color_image, (u, v), 5, (0, 0, 255), -1)
        cv2.putText(
            color_image,                   
            f"Depth: {distance:.3f} m",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

        cam.imshow("color_image", color_image)
        cam.imshow("depth_image", depth_colorMap)

        key = cam.waitKey(1)
        if key == 27:
            break

finally:
    cam.stopPipeline()
    cam.destroyAllWindows()

[INFO] [Head D435i] Pipeline started.


[INFO] [Head D435i] Pipeline stopped.


<span style="color:#B8860B; font-weight:bold; font-size:50px;">
3D position of center pixel based on Camera Reference Frame
</span>

In [ ]:
# --- Select camera location ----------------------------
location = "head"       # head / wrist_R / wrist_L
# -------------------------------------------------------

if location == "head":
    cam = d435i
elif location == "wrist_R":
    cam = d405_R
elif location == "wrist_L":
    cam = d405_L

# Show center 3D point position [x,y,z] in camera frame
cam.startPipeline()

try:
    while True:
        color_frame, depth_frame = cam.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = cam.frameToImage(color_frame, depth_frame)
        depth_colorMap = cam.convertDepthToColorMap(depth_image, alpha=0.03)

        h, w, _ = color_image.shape
        u = w // 2      # origin is left    
        v = h // 2      # origin is top 

        pos = cam.get3DPos(depth_frame, (u,v))
        X, Y, Z = pos

        cv2.circle(color_image, (u, v), 5, (0, 0, 255), -1)
        cv2.putText(
            color_image,
            f"X:{X:.3f}, Y:{Y:.3f}, Z:{Z:.3f} m",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

        cam.imshow("color_image", color_image)
        cam.imshow("depth_image", depth_colorMap)

        key = cam.waitKey(1)
        if key == 27:
            break

finally:
    cam.stopPipeline()
    cam.destroyAllWindows()

[INFO] [Right Wrist D405] Pipeline started.
[INFO] [Right Wrist D405] Pipeline stopped.


<span style="color:#B8860B; font-weight:bold; font-size:50px;">
3D position of clicked point 
</span>

In [12]:
# --- Select camera location ----------------------------
location = "head"       # head / wrist_R / wrist_L
# -------------------------------------------------------

if location == "head":
    cam = d435i
elif location == "wrist_R":
    cam = d405_R
elif location == "wrist_L":
    cam = d405_L

cam.openWindowWithMouse("color_image")

cam.startPipeline()

try:
    while True:
        color_frame, depth_frame = cam.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = cam.frameToImage(color_frame, depth_frame)
        depth_colorMap = cam.convertDepthToColorMap(depth_image, alpha=0.03)

        if cam.clicked_point is not None:
            u, v = cam.clicked_point

            X, Y, Z = cam.get3DPos(depth_frame, (u,v))

            cv2.circle(color_image, (u, v), 5, (0, 0, 255), -1)
            cv2.putText(
                color_image,
                f"X:{X:.3f}, Y:{Y:.3f}, Z:{Z:.3f} m",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2
            )

            print(f"Pixel=({u}, {v}), Camera 3D=({X:.3f}, {Y:.3f}, {Z:.3f}) m")

        cam.imshow("color_image", color_image)
        cam.imshow("depth_image", depth_colorMap)

        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    cam.stopPipeline()
    cam.destroyAllWindows()

[INFO] [Head D435i] Pipeline started.
Clicked pixel: u=407, v=232
Pixel=(407, 232), Camera 3D=(0.546, -0.080, 3.384) m
Pixel=(407, 232), Camera 3D=(0.549, -0.080, 3.403) m
Pixel=(407, 232), Camera 3D=(0.546, -0.080, 3.384) m
Pixel=(407, 232), Camera 3D=(0.552, -0.081, 3.422) m
Pixel=(407, 232), Camera 3D=(0.534, -0.078, 3.312) m
Pixel=(407, 232), Camera 3D=(0.534, -0.078, 3.312) m
Pixel=(407, 232), Camera 3D=(0.540, -0.079, 3.348) m
Pixel=(407, 232), Camera 3D=(0.552, -0.081, 3.422) m
Pixel=(407, 232), Camera 3D=(0.523, -0.077, 3.242) m
Pixel=(407, 232), Camera 3D=(0.534, -0.078, 3.312) m
Pixel=(407, 232), Camera 3D=(0.540, -0.079, 3.348) m
Pixel=(407, 232), Camera 3D=(0.531, -0.078, 3.294) m
Pixel=(407, 232), Camera 3D=(0.520, -0.076, 3.225) m
Pixel=(407, 232), Camera 3D=(0.543, -0.079, 3.366) m
Pixel=(407, 232), Camera 3D=(0.549, -0.080, 3.403) m
Pixel=(407, 232), Camera 3D=(0.531, -0.078, 3.294) m
Pixel=(407, 232), Camera 3D=(0.549, -0.080, 3.403) m
Pixel=(407, 232), Camera 3D=(0.53

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Center 3D position of Colored Object (2D BBox)
</span>

In [14]:
# --- Select camera location ----------------------------
location = "head"       # head / wrist_R / wrist_L
# -------------------------------------------------------

if location == "head":
    cam = d435i
elif location == "wrist_R":
    cam = d405_R
elif location == "wrist_L":
    cam = d405_L

cam.startPipeline()

try:
    while True:
        color_frame, depth_frame = cam.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = cam.frameToImage(color_frame, depth_frame)
        depth_colorMap = cam.convertDepthToColorMap(depth_image, alpha=0.03)

        detected_objects, color_image, mask = cam.detectColorObjectContour(
            color_image=color_image,
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_BLACK,
            minimum_area=300                    # Can be changed
        )

        for obj in detected_objects:
            print(
                f"[object {obj['id']}] "
                f"pixel={obj['pixel']}, "
                f"pos={np.round(obj['pos'], 3)}, "
                f"area={obj['area']:.1f}"
            )

        cam.imshow("color_image", color_image)
        cam.imshow("depth_image", depth_colorMap)
        cam.imshow("mask", mask)

        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    cam.stopPipeline()
    cam.destroyAllWindows()

[INFO] [Head D435i] Pipeline started.
[object 7] pixel=[320, 300], pos=[0.064 0.313 3.54 ], area=218605.5
[object 21] pixel=[353, 78], pos=[ 0.628 -2.404  8.675], area=729.0
[object 32] pixel=[274, 30], pos=[-0.46  -2.848  7.999], area=425.5
[object 35] pixel=[105, 57], pos=[-3.086 -2.865  9.193], area=9541.5
[object 5] pixel=[320, 303], pos=[0.064 0.33  3.54 ], area=173264.0
[object 15] pixel=[40, 33], pos=[-3.84  -3.046  8.675], area=917.5
[object 15] pixel=[320, 303], pos=[0.064 0.33  3.54 ], area=163291.5
[object 21] pixel=[44, 33], pos=[-3.783 -3.046  8.675], area=439.5
[object 16] pixel=[123, 323], pos=[-2.218  0.914  7.246], area=9440.5
[object 19] pixel=[320, 303], pos=[0.064 0.33  3.54 ], area=112081.5
[object 25] pixel=[44, 33], pos=[-3.783 -3.046  8.675], area=327.5
[object 10] pixel=[160, 429], pos=[-0.987  1.21   4.026], area=9127.0
[object 11] pixel=[377, 377], pos=[0.644 1.238 5.757], area=503.0
[object 27] pixel=[142, 370], pos=[-1.676  1.241  6.099], area=1500.0
[objec

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Center 3D position of Colored Object (3D BBox)
</span>

In [16]:
# --- Select camera location ----------------------------
location = "head"       # head / wrist_R / wrist_L
# -------------------------------------------------------

if location == "head":
    cam = d435i
elif location == "wrist_R":
    cam = d405_R
elif location == "wrist_L":
    cam = d405_L

cam.startPipeline()

try:
    while True:
        color_frame, depth_frame = cam.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = cam.frameToImage(color_frame, depth_frame)
        depth_colorMap = cam.convertDepthToColorMap(depth_image, alpha=0.03)

        detected_objects, color_image, mask = cam.detectColorObjectContour3D(
            color_image=color_image,
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_BLACK,       # Can be changed
            minimum_area=300                    # Can be changed
        )

        for obj in detected_objects:
            print(
                f"[object {obj['id']}] "
                f"center_3d={np.round(obj['center_3d'], 3)}, "
                f"bbox_3d_size={np.round(obj['bbox_3d_size'], 3)}"
            )

        cam.imshow("color_image", color_image)
        cam.imshow("depth_image", depth_colorMap)
        cam.imshow("mask", mask)

        key = cam.waitKey(1)
        if key == 27:
            break

finally:
    cam.stopPipeline()
    cam.destroyAllWindows()

[INFO] [Head D435i] Pipeline started.
[object 0] center_3d=[-0.076  0.411  4.106], bbox_3d_size=[11.867  8.158 24.979]
[object 1] center_3d=[-0.417 -2.866  8.555], bbox_3d_size=[0.344 0.591 0.845]
[object 2] center_3d=[-3.555 -2.994  8.555], bbox_3d_size=[1.428 2.113 3.312]
[object 0] center_3d=[-0.547  1.318  3.779], bbox_3d_size=[1.601 0.676 0.874]
[object 1] center_3d=[-0.793  0.977  4.19 ], bbox_3d_size=[0.334 0.206 0.368]
[object 2] center_3d=[-1.725  1.221  5.98 ], bbox_3d_size=[0.349 0.374 0.866]
[object 3] center_3d=[-2.365  0.819  6.844], bbox_3d_size=[0.714 0.564 1.413]
[object 4] center_3d=[0.732 0.086 3.48 ], bbox_3d_size=[11.049  8.134 24.964]
[object 0] center_3d=[-0.548  1.313  3.779], bbox_3d_size=[1.607 0.676 0.874]
[object 1] center_3d=[-0.794  0.977  4.19 ], bbox_3d_size=[0.338 0.217 0.368]
[object 2] center_3d=[-1.728  1.219  5.98 ], bbox_3d_size=[0.382 0.403 0.866]
[object 3] center_3d=[-2.388  0.823  6.921], bbox_3d_size=[0.701 0.581 1.413]
[object 4] center_3d=[0